In [0]:
%load_ext autoreload
%autoreload 2

import sys, os
import warnings

sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

# iFood Offer Personalization — Propensity Modeling & Business Impact

### Introduction

This notebook documents the development and evaluation of a propensity model designed to optimise offer distribution strategy. Building upon the behavioral and demographic signals identified in the exploratory data analysis, the goal is to predict the likelihood of a customer completing a given offer — creating a data-driven tool that supports the decision of **which offer to send to which customer**, with the objective of increasing conversion rates and maximising the financial return on marketing investments.

This notebook covers the feature engineering process, presents model performance on a holdout test set, and translates these results into a tangible business impact through financial simulation.

---

### Feature Engineering

The modeling dataset is structured at the **(customer × offer × moment)** grain — one row per offer opportunity. Features were engineered with strict point-in-time correctness to prevent data leakage:

* **Pre-campaign spending behavior:** `avg_ticket_before`, `total_spend_before`, `tx_count_7d`, `tx_count_28d` — capturing spend intensity in the 7 and 28 days prior to offer receipt.
* **Historical offer engagement:** `offers_viewed_count_before`, `offers_completed_count_before`, `customer_conversion_rate_before`, `view_rate_before` — encoding how the customer has responded to past campaigns.
* **Recency signals:** `days_since_last_tx`, `days_since_last_offer_view` — the strongest behavioral timing signals identified in the EDA.
* **Customer profile:** `customer_tenure_days`, `customer_age`, `customer_gender`, `customer_credit_limit` — demographic and loyalty context.
* **Offer characteristics:** `offer_type`, `offer_duration_days`, `offer_discount_value`, `offer_min_spend`, `offer_reward_ratio`, `offer_duration_bucket` — structural attributes of the offer being distributed.
* **Channel flags:** `channel_is_email`, `channel_is_mobile`, `channel_is_social`, `channel_is_web`, `offer_channel_count` — binary indicators of distribution channel.

---

### Model Performance

The trained LightGBM model demonstrates strong predictive power on the holdout test set:

* **Discrimination:** ROC-AUC of **0.8527** (+0.35 over stratified baseline) — the model ranks a random converter above a random non-converter 85% of the time.
* **Positive class performance:** PR-AUC of **0.8593** (+0.30 over baseline) — the primary evaluation metric for this imbalanced task, confirming the model learns real signal beyond the natural conversion rate.
* **Score separation:** KS statistic of **0.548** at threshold 0.663 — converters and non-converters are well-separated across the score distribution, validating the decile targeting strategy.
* **Key conversion drivers (SHAP):** `customer_tenure_days` and `avg_ticket_before` are the top predictors — behavioral and loyalty signals dominate over offer characteristics. `offer_discount_value` ranks 3rd but with a counterintuitive negative direction: high discounts paired with high minimum spend reduce completion probability.

---

### Decision Policy, Simulation & Financial Uplift

* **Profit-maximizing threshold:** the optimal threshold is **0.06** — equal to the analytical break-even point (`OFFER_COST / avg_conversion_value` = 1.00 / 16.62), generating an expected profit of **BRL 250,589** on the test set vs. **BRL 248,469** for a send-to-all baseline (+BRL 2,120 | +0.9%).
* **Efficiency gain:** the model excludes **3,494 of 33,988 opportunities (10.3%)** — those with the lowest predicted propensity — with minimal impact on captured conversions. At iFood's scale of 55 million active users, systematically deprioritising low-propensity sends compounds into a structural improvement in campaign ROI.
* **Budget simulation (BRL 10,000):** targeting the top 10,000 scored opportunities delivers **7,852 conversions** at **78.5% precision** and a **lift of 1.41×** — an expected ROI of **13.26×** on campaign spend.
* **Decile strategy:** deciles 1–3 capture **47% of all conversions** while targeting only **30% of the base** — the recommended operating point for standard campaigns.

> **Note on financial figures:** `OFFER_COST` (BRL 1.00) and `avg_conversion_value` (BRL 16.62, mean historical ticket) are proxies. ROI estimates are directional — validate with Finance before operational budget decisions.


## 2. Setup & Imports

In [0]:
# Uncomment to install dependencies (run once per cluster restart)
# %pip install lightgbm shap scikit-learn optuna optuna-integration[sklearn]

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

from ifood_case.feature_engineering import build_abt, LookbackConfig
from ifood_case.model_trainer import LGBMTrainer
from ifood_case.evaluator import Evaluator
from ifood_case.utils import plot_correlation_matrix, plot_feature_distributions, plot_financial_uplift, display_policy_report
from ifood_case.data_quality import key_check, null_summary

RANDOM_STATE = 7

## 3. Business Parameters

`OFFER_COST` represents the estimated cost per offer sent (e.g., SMS delivery + marketing ops overhead).
This is a synthetic premise — replace with the actual cost from the finance/ops team before production.

`avg_conversion_value` will be derived from training data (mean ticket of converters) to avoid arbitrary assumptions.

In [0]:
OFFER_COST = 1.00  # BRL: estimated cost per offer send (hypothetical and synthetic)

## 4. Load Modeling Dataset (ABT)

In [0]:
df_joined     = spark.table("default.ifood_case_df_joined")
opps_enriched = spark.table("default.ifood_case_opps_enriched")

LOOKBACK = LookbackConfig(days_short=7.0, days_long=28.0)

abt_df, numerical_columns, categorical_columns = build_abt(
    df_joined=df_joined,
    opps_enriched=opps_enriched,
    lookback=LOOKBACK,
)

n_rows = abt_df.count()
print(f"ABT: {n_rows:,} rows × {len(abt_df.columns)} columns")
print(f"Numerical  ({len(numerical_columns)}): {numerical_columns}")
print(f"Categorical ({len(categorical_columns)}): {categorical_columns}")
display(abt_df.limit(3))

## 5. Data Quality Validation

Before modeling, we validate ABT integrity:
- **Grain uniqueness**: each (customer, offer, time) must appear exactly once
- **Null rates**: NULLs are expected for customers with no history (e.g., `avg_ticket_before`). Flag unexpected nulls.
- **Class balance**: informs metric choice and potential class weighting strategy

#### 5.1 Grain uniqueness

In [0]:
grain_check = key_check(abt_df, key_cols=["customer_id", "offer_id", "time_received"])
print("Grain check:", grain_check)
assert grain_check["duplicate_key_groups"] == 0, "ABT grain is NOT unique — investigate before modeling!"

#### 5.2 Null rates per feature

In [0]:
display(null_summary(abt_df, cols=numerical_columns + categorical_columns))

#### 5.3 Class balance

In [0]:
display(
    abt_df.groupBy("target").count()
    .withColumn("pct", F.round(F.col("count") / F.lit(n_rows), 4))
    .orderBy("target")
)

## 6. Feature Diagnostics

#### 6.1 Correlation Matrix

In [0]:
plot_correlation_matrix(df=abt_df, numerical_cols=numerical_columns, target_col="target")

**Key correlation insights**

- **Historical responsiveness dominates:** The strongest correlations with `target` are observed for `customer_conversion_rate_before`, `offers_completed_count_before`, and `offers_viewed_count_before`, indicating that past response to campaigns is the most informative signal for predicting future offer completion.

- **Two behavioral dimensions emerge:** The matrix reveals two correlated clusters: (1) purchasing intensity (`total_spend_before`, `transaction_count_before`) and (2) promotional engagement (`offers_viewed_count_before`, `offers_completed_count_before`). This suggests that conversions may arise from both **high platform activity** and **promotion-seeking behavior**.

- **Offer attributes have limited standalone power:** Variables describing the offer itself (`offer_discount_value`, `offer_min_spend`, `offer_duration_days`) show relatively weak correlation with the target, suggesting that **customer behavior is more predictive than the offer configuration alone**.

- **Correlated behavioral signals:** Spending and activity variables are strongly correlated, reflecting a latent engagement dimension. While redundant for linear models, tree-based algorithms such as LightGBM can leverage these complementary signals.

- **Evidence of non-linear drivers:** The absence of very high correlations with the target suggests that conversion is driven by **interactions between behavioral features**, reinforcing the choice of gradient boosting models.

- **Channel strategy signal:** The negative correlation between `channel_is_mobile` and `offer_duration_days` indicates that mobile campaigns tend to have shorter validity windows, consistent with push-based engagement strategies.

### 6.2 Feature Distributions by Target

Box plots of key features split by target class.
Features with clearly separated distributions will drive the most model signal — anticipating the SHAP analysis.

In [0]:
plot_feature_distributions(
    df=abt_df,
    numerical_cols=numerical_columns,
    target_col="target",
    top_n=12,
)

## 7. Train/Test Strategy

### Split Design

| Set | Selection rule | Size | Purpose |
|-----|---------------|------|---------|
| **Test** | Last offer per customer | ~1 per customer | Final evaluation (no peeking during training) |
| **Train** | All prior offers | Remaining rows | Model fitting |
| **Calibration** | 20% of train (group-safe) | ~20% of train | Isotonic calibration + threshold selection |

**Last-event split**: simulates real deployment where the model is trained on a customer's history and predicts the outcome of their next offer. A random row split would leak future information across customers.

**Group-safe calibration split**: ensures the same customer does not appear in both the calibration training and calibration test, preventing identity-based overfitting in the probability calibrator.

**GroupKFold in CV**: prevents a customer's multiple offer rows from appearing in different CV folds simultaneously, which would inflate cross-validation scores.

In [0]:
trainer = LGBMTrainer(
    df=abt_df,
    numerical_columns=numerical_columns,
    categorical_columns=categorical_columns,
    target="target",
    id_col="customer_id",
    time_col="time_received",
    random_state=RANDOM_STATE,
)
x_train, x_test, y_train, y_test = trainer.train()

print(f"Train : {len(x_train):,} rows  |  conversion rate: {y_train.mean():.2%}")
print(f"Test  : {len(x_test):,} rows  |  conversion rate: {y_test.mean():.2%}")

## 8. Baseline Model

A stratified dummy classifier provides the **performance floor** — a model that predicts randomly while respecting class proportions.
Any meaningful model must significantly exceed these scores.
Without a baseline, metrics like "ROC-AUC = 0.85" are harder to contextualize.

In [0]:
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
dummy.fit(x_train, y_train)
y_proba_baseline = dummy.predict_proba(x_test)[:, 1]

baseline_roc = roc_auc_score(y_test, y_proba_baseline)
baseline_pr  = average_precision_score(y_test, y_proba_baseline)

print(f"Baseline ROC-AUC : {baseline_roc:.4f}  (expected ~0.50 for random)")
print(f"Baseline PR-AUC  : {baseline_pr:.4f}  (expected ~base_rate = {y_test.mean():.2f})")
print("\nLGBM must beat both metrics significantly to demonstrate real predictive value.")

## 9. LightGBM Model Training

Hyperparameters tuned via **Optuna** (50 trials) with **GroupKFold** cross-validation.
Scoring metric: `average_precision` (PR-AUC on validation fold).
Best model is then calibrated with isotonic regression on the held-out calibration set.

In [0]:
print(f"Best CV PR-AUC  : {trainer.best_score:.4f}")
print(f"Best parameters : {trainer.best_params}")

## 10. Model Evaluation

### 10.1 Threshold-Independent Metrics

These metrics evaluate the model's ranking quality regardless of the classification threshold:
- **ROC-AUC**: probability that the model ranks a random positive above a random negative
- **PR-AUC**: area under precision-recall curve — primary metric for this imbalanced task
- **LogLoss**: penalizes confident wrong predictions — relevant when probabilities are used for ROI
- **Brier score**: mean squared error of predicted probabilities — measures calibration quality

In [0]:
y_proba_test = trainer.predict_proba(x_test)
e_test = Evaluator(y_true=y_test)

quality = e_test.model_quality(y_proba_test)
print(quality)

print(f"\nLGBM vs Stratified Baseline:")
print(f"  ROC-AUC : {quality.roc_auc:.4f}  vs  {baseline_roc:.4f}  (+{quality.roc_auc - baseline_roc:.4f})")
print(f"  PR-AUC  : {quality.pr_auc:.4f}  vs  {baseline_pr:.4f}  (+{quality.pr_auc - baseline_pr:.4f})")

### 10.2 ROC-AUC Curve

In [0]:
e_test.plot_roc_curve(y_proba_test)

### 10.3 Precision-Recall Curve

The PR curve is the **primary diagnostic** for this task.
A PR-AUC substantially above the base rate (natural conversion rate) confirms the model is learning real signal.
The curve shows the precision/recall trade-off as the decision threshold varies — useful for selecting operating points.

In [0]:
e_test.plot_pr_curve(y_proba_test)

#### Curva ROC and Curva Precision-Recall

##### ROC Curve: Model Discrimination Ability

* **AUC = 0.8527 vs. Baseline = 0.5034 (+0.3493):** There is an 85.27% probability that the model ranks a random converter above a random non-converter — strong discriminative power, significantly above a random guess.
* **Curve Shape:** The curve rises steeply toward the top-left corner before diverging right, indicating the model captures the majority of true converters with very few false alarms at high score thresholds — ideal for budget-constrained campaigns where only top-ranked customers are targeted.
* **Gain over baseline:** The +0.35 lift over the stratified baseline confirms the model learns real behavioral patterns, not just the dataset's natural conversion rate.

> **Business Insight:** An AUC of 0.85 means the model is highly effective at ranking customers by conversion likelihood, providing a strong foundation for a score-based targeting strategy.

##### Precision-Recall Curve: Performance on the Positive Class

* **PR-AUC = 0.8593 vs. Baseline = 0.56 (+0.2990):** The model substantially outperforms the no-skill baseline (the natural conversion rate), confirming it adds real predictive value beyond simply predicting the majority class.
* **Curve Shape:** Precision remains high (~0.85+) across a wide range of recall values before dropping sharply near recall=1.0 — meaning the model can recover the vast majority of converters while maintaining strong hit rates.
* **Why this is the primary metric:** Unlike ROC-AUC, PR-AUC is sensitive to the cost of false positives. In a campaign context, sending an offer to a non-converter has a direct cost. PR-AUC directly penalizes this error, making it the more meaningful metric for profitability analysis.

> **Business Insight:** A PR-AUC of 0.86 means the model is not just accurate overall — it is specifically effective at the task that drives business value: identifying customers who will complete offers without requiring mass, untargeted distribution.




### 10.4 Confusion Matrix (Default Threshold = 0.5)

The default threshold of 0.5 is a useful diagnostic but rarely the optimal business threshold.
The profit-maximizing threshold (Section 12) will yield a different operating point.

In [0]:
e_test.plot_confusion_matrix(y_proba_test, threshold=0.5)

report_05 = e_test.classification_report_at_threshold(y_proba_test, threshold=0.5)
pd.DataFrame(report_05).T

#### Confusion Matrix: Classification Accuracy at Default Threshold

* **True Positives (8,484 | 49.92%):** The model correctly identified 8,484 customers who completed the offer — the primary value driver, representing nearly half of the entire test set.
* **True Negatives (4,664 | 27.44%):** 4,664 non-converters correctly excluded from targeting — direct cost savings by avoiding unnecessary offer sends.
* **False Positives (2,836 | 16.69%):** Offers sent to customers who would not convert. This is efficiency lever: reducing FPs through threshold optimization (Block 12) directly improves campaign ROI.
* **False Negatives (1,010 | 5.94%):** Missed opportunities — converters the model did not identify. At 5.94%, this is acceptably low, ensuring minimal revenue left on the table.

**Key metrics at threshold 0.50 — Class 1 (converters):**
- **Recall: 89.36%** — the model finds 9 out of every 10 customers who would convert.
- **Precision: 74.96%** — when the model recommends targeting, it is correct ~75% of the time.

> The default threshold of 0.50 prioritizes **recall over precision** — appropriate for a growth-oriented strategy. For a profit-maximizing strategy, the optimal threshold is derived in Block 12.


### 10.5 Calibration Curve

A well-calibrated model outputs probabilities that match observed frequencies:
if the model says p=0.3, roughly 30% of those customers should convert.

**Why calibration matters for this use case**: the threshold and budget optimization in Section 12 use
the formula `p * avg_conversion_value - cost`. If probabilities are systematically over- or under-estimated,
the ROI calculation will be biased. Isotonic calibration was applied during training to correct this.

In [0]:
e_test.plot_calibration_curve(y_proba_test, n_bins=10, strategy="quantile")

### 10.6 KS Statistic

The **Kolmogorov-Smirnov (KS) statistic** measures the maximum separation between the
cumulative score distributions of converters and non-converters.

- KS > 0.40: strong score separation — reliable for ranking-based targeting
- The score threshold at the KS maximum is the point where the model discriminates best between the two classes

In [0]:
ks_val, ks_thr = e_test.ks_statistic(y_proba_test)
print(f"KS Statistic : {ks_val:.4f}")
print(f"At score threshold : {ks_thr:.4f}")

e_test.plot_ks(y_proba_test)

#### KS Statistic: Score Separation Between Converters and Non-Converters

* **KS = 0.548 at score threshold 0.663:** At a score of ~0.66, the cumulative distribution of converters is 54.8 percentage points ahead of non-converters. This is the point of maximum separation — where the model most cleanly distinguishes the two populations.
* **Interpretation:** A KS above 0.40 is generally considered strong for a ranking model. KS = 0.55 confirms that the model produces a meaningful score gradient: high scores concentrate converters, low scores concentrate non-converters.
* **Business relevance:** This directly validates the decile strategy (Block 11). Customers ranked in the top deciles are disproportionately converters — the KS curve visually shows how quickly the gap opens as we move down the score ranking.


## 11. Ranking Evaluation

In a propensity use case the model is typically used for **ranking**, not binary classification.
The marketing team targets the top-K customers, not everyone above a probability threshold.
Ranking metrics directly translate model quality into campaign design decisions.

### 11.1 Lift & Cumulative Gains Curves

- **Gains**: what fraction of all converters do we capture if we target the top X% of scored customers?
- **Lift**: how many times better than random is our top X% bucket?

These curves directly answer the business question: *"If we can only send offers to N% of customers, which N% should we pick?"*

In [0]:
e_test.plot_gains_lift(y_proba_test)

#### Cumulative Gains & Lift: Ranking Efficiency

**Cumulative Gains:**
* **Top 20% targeted → ~38% of converters captured:** By sending offers to only 1 in 5 customers (highest-scored), the model already recovers more than a third of all potential converters — nearly **2× what random targeting would achieve** for the same budget.
* **Top 40% → ~65% of converters:** With less than half the base contacted, the model identifies two-thirds of the total conversion opportunity available.
* **Top 60% → ~85% of converters:** Beyond this point, marginal gains diminish sharply — a clear signal of where campaign budget should be concentrated for maximum efficiency.

**Lift:**
* **Initial Lift (~1.78×):** Among the highest-scored customers, the conversion rate is 1.78× the base rate — targeting the top 10% of the ranked list yields proportionally 78% more converters than random targeting of the same population fraction.
* **Lift at top 20% (~1.70×):** Still 70% above the random baseline, confirming the model maintains strong ranking efficiency even when expanding beyond the most selective tier.
* **Smooth monotonic decay:** Lift decreases consistently as the targeted population grows — expected and healthy behavior with no instability.

> **Business Insight:** Lift is a pure ranking metric — it measures how many more converters the model finds compared to random, for the same targeting budget. The financial return of acting on this ranking is quantified in Block 12 (Profit Curve & Decision Policy).



### 11.2 Decile Analysis

The scored population is divided into 10 equally-sized buckets, ranked by predicted probability (decile 1 = highest scores).

**Business use**: this table is the primary deliverable for the marketing team.
It translates the model into a simple rule: *"Target deciles 1–3 for the best ROI."*
A good model shows monotonically decreasing conversion rates from decile 1 to decile 10.

In [0]:
decile_df = e_test.decile_table(y_proba_test)
display(
    decile_df.style
    .background_gradient(subset=["lift", "conversion_rate"], cmap="RdYlGn")
    .format({"avg_score": "{:.3f}", "conversion_rate": "{:.2%}",
              "lift": "{:.2f}x", "cumulative_recall": "{:.2%}"})
)

#### Key Insights

* **Decile 1 (94.53% | lift 1.69×):** Near-certain conversion — optimal for high-cost or high-value offers. Avg score 0.98.
* **Deciles 1–3 capture 47.62% of all converters targeting only 30% of the base** — half the opportunity at a third of the budget.
* **Decile 6 = break-even (lift 0.99×):** Below this point the model adds no value over random targeting. Deciles 7–10 convert below the base rate — **do not target**.
* **Decile 10: only 3.46% conversion rate** — the model actively identifies non-converters, which is as strategically valuable as identifying converters.

| Tier | Deciles | Conversion | Action |
|---|---|---|---|
| High priority | 1–3 | 83–95% | Always target |
| Budget-dependent | 4–5 | 69–77% | Include if cost allows |
| Exclude | 6–10 | ≤56% | Do not send offer |

> **Natural next step:** segmented models per `offer_type` — bogo, discount, and informational have distinct conversion mechanics and can benefit from independent targeting policies.






### 11.3 Precision & Recall @K

Point estimates at specific targeting volumes:
- **Precision@K**: of the top-K customers we target, what fraction actually converts?
- **Recall@K**: of all converters in the test set, what fraction falls within the top K?
- **Lift@K**: precision@K / base_rate

In [0]:
rank_df = e_test.ranking_table(y_proba_test, ks=[100, 1_000, 2000, 5_000, 10000, 16994])
display(
    rank_df.style
    .background_gradient(subset=["precision_at_k", "lift_at_k"], cmap="RdYlGn")
    .background_gradient(subset=["recall_at_k"], cmap="Blues")
    .format({
        "k_rate":         "{:.2%}",
        "precision_at_k": "{:.2%}",
        "recall_at_k":    "{:.2%}",
        "lift_at_k":      "{:.2f}x",
    })
)

#### Key Insights

| k | % of base | Precision | Recall | Recommended scenario |
|---|---|---|---|---|
| 1,000 | 5.88% | 95.5% | 10% | Premium offer, high contact cost |
| 5,000 | 29.42% | 88.8% | 47% | Standard campaign |
| 10,000 | 58.84% | 78.5% | 83% | Maximize reach |

* **k=1,000 (top 5.88%) → Precision 95.5%, Recall 10%:** With less than 6% of the base, the model delivers 955 converters at a 95.5% hit rate. Best suited for high-cost-per-contact campaigns where every send must count.

* **k=5,000 (top 29.42%) → Precision 88.76%, Recall 47%:** Targeting 30% of the base, the model captures nearly half of all converters while maintaining 89% precision. **Recommended balance point for volume campaigns.**

* **k=10,000 (top 58.84%) → Precision 78.52%, Recall 83%:** Contacting 59% of the base captures 83% of all converters. Lift still positive (1.41×) — suited for reach-maximization strategies where contact cost is low.

> **Business insight:** this table translates the model into budget planning — the business team selects the K that balances contact cost and captured opportunity without needing to understand the model internals.




## 12. Decision Policy Optimization

Converting model scores into a targeting decision requires choosing a **policy**:
1. **Threshold policy**: target all customers with score ≥ threshold
2. **Budget policy**: target top-K customers given a fixed budget (K = budget / cost_per_contact)

Both policies are evaluated. The threshold is selected to maximize expected profit — **not** at the default 0.5.

### 12.1 Derive avg_conversion_value from Data

Rather than assuming a fixed conversion value, I derive it from the training set:
the mean `avg_ticket_before` of customers who converted.
This is a conservative proxy — it uses historical ticket size, not revenue per conversion event.

In [0]:
avg_conversion_value = float(
    x_train.assign(_target=y_train.values)
    .query("_target == 1")["avg_ticket_before"]
    .dropna()
    .mean()
)

print(f"avg_conversion_value (mean ticket of converters in train) : BRL {avg_conversion_value:.2f}")
print(f"OFFER_COST (premise)                                      : BRL {OFFER_COST:.2f}")
print(f"Break-even threshold (analytical)                         : {OFFER_COST / avg_conversion_value:.4f}")

### 12.2 Profit Curve

Expected profit as a function of the decision threshold.
The curve reveals the profit peak and confirms that deviating from the default 0.5 threshold is justified.

Formula per targeted customer: `p_i × avg_conversion_value − OFFER_COST`
Sum over all customers where p_i ≥ threshold.

In [0]:
e_test.plot_profit_curve(
    y_pred_proba=y_proba_test,
    avg_conversion_value=avg_conversion_value,
    cost_per_contact=OFFER_COST,
)

#### Key Insights
- The profit curve peaks at threshold **0.06** (BRL 158,760), which equals the analytical break-even point `OFFER_COST / avg_conversion_value` = `1.00 / 16.62` — model and theory align.
- The low optimal threshold reflects the economics of this campaign: at BRL 1.00 per contact and BRL 16.62 average ticket, virtually any customer with measurable conversion propensity is worth targeting.
- Profit declines monotonically beyond 0.06 — raising the threshold increases precision but sacrifices volume, reducing total expected return. The right threshold is a **business decision**, not a modeling one: it depends on the actual budget constraint and contact cost.
- The curve never crosses zero — even at high thresholds the campaign remains profitable, confirming the model produces no harmful targeting recommendations.


### 12.3 Optimal Threshold Selection



In [0]:
x_calib, y_calib = trainer.calibration_set
y_proba_calib = trainer.predict_proba(x_calib)
e_calib = Evaluator(y_true=y_calib)

best_thr, best_profit_calib = e_calib.find_threshold_max_expected_profit(
    y_pred_proba=y_proba_calib,
    avg_conversion_value=avg_conversion_value,
    cost_per_contact=OFFER_COST,
)

print(f"Optimal threshold (calibration set) : {best_thr:.4f}")
print(f"Expected profit on calibration set   : BRL {best_profit_calib:,.2f}")

> **Methodological note:** the profit of BRL 158,760 (Block 12.2) is computed on the 
> test set and serves to communicate financial impact. The threshold of 0.06 (Block 12.3) 
> is selected on the calibration set to avoid evaluation leakage — the two figures are 
> not directly comparable as they are derived from different datasets.



### 12.4 Apply Optimal Threshold on Test Set

In [0]:
e_test.plot_confusion_matrix(y_proba_test, threshold=best_thr)

report_opt = e_test.classification_report_at_threshold(y_proba_test, threshold=best_thr)
print(f"Classification report at threshold = {best_thr:.4f}:")
display(pd.DataFrame(report_opt).T)

policy = e_test.policy_threshold(
    y_pred_proba=y_proba_test,
    threshold=best_thr,
    avg_conversion_value=avg_conversion_value,
    cost_per_contact=OFFER_COST,
    policy_name=f"Threshold={best_thr:.2f} (max profit)",
)
print("\nPolicy report:")
print(policy)


#### Key insights

- At the profit-maximizing threshold of **0.06**, the model targets 15,519 customers (91.3% of the base) — capturing **99.6% of all converters** (recall) while accepting a precision of 60.9%.
- Only **41 converters are missed** (FN = 0.24%) — the model's primary goal at this threshold is coverage, not selectivity.
- The trade-off is intentional: with a contact cost of BRL 1.00 and an average ticket of BRL 16.62, the expected value of reaching a converter far exceeds the cost of a false positive — making high recall the economically rational operating point.
- For campaigns with higher contact costs or lower average tickets, a more restrictive threshold would shift the balance toward precision — this is the key lever for adapting the policy to different business contexts.


### 12.5 Budget Simulation

Fixed-budget targeting: given a campaign budget of BRL 10,000 and a cost of BRL 1.00 per offer,
the model selects the top ~10,000 customers by predicted probability.

This policy is useful when the constraint is budget rather than a probability cutoff.

In [0]:
policy_budget = e_test.policy_topk_by_budget(
    y_pred_proba=y_proba_test,
    budget_brl=10_000,
    cost_per_contact=OFFER_COST,
    avg_conversion_value=avg_conversion_value,
    policy_name="Top-K by BRL 10k budget",
)

display(display_policy_report(policy_budget))


#### Key insights

**Budget Simulation: BRL 10,000**

With a fixed budget of BRL 10,000 and a cost of BRL 1.00 per offer, the model automatically
selects the top 10,000 highest-scored customers to receive the offer.

* **Customers targeted:** 10,000 (58.8% of base) — the model prioritizes the most likely converters within the available budget
* **Expected conversions:** 7,852 out of 10,000 contacts convert — **precision of 78.5%**
* **Recall of 82.7%:** targeting just over half the base, the campaign captures 83% of all possible converters
* **Lift of 1.41×:** each BRL invested in this group generates 41% more conversions than a random targeting of the same size
* **Expected profit: BRL 132,572** on a spend of BRL 10,000
* **Expected ROI: 13.26×** — for every BRL 1.00 invested, the model returns BRL 13.26 in transacted value

> **Limitation:** `expected_profit` uses `avg_ticket_before` as a conversion value proxy
> and does not discount the offer's `discount_value`. Actual ROI will be lower — this figure
> is best used for relative comparison between policies, not as an absolute revenue estimate.


## 13. Financial Uplift

In [0]:
plot_financial_uplift(
    y_pred_proba=y_proba_test,
    avg_conversion_value=avg_conversion_value,
    cost_per_contact=OFFER_COST,
    optimal_threshold=best_thr,
)


##### Key insights
- The model reduces offer distribution volume by **10.3%** while retaining **99.6% of all conversions** — meaning 1 in every 10 offers currently sent adds no incremental value and can be eliminated without meaningful revenue impact.
- At iFood's scale of  +**55 million active users**, this precision translates directly into campaign efficiency: fewer touches on low-propensity customers, more budget concentrated on those most likely to convert.
- **The strategic case is not about saving on a single campaign — it is about redefining the baseline.** A model that consistently removes 10% of wasted sends across every offer type, every wave, and every user cohort compounds into a structural improvement in marketing ROI that grows with the platform.


> **Note on financial figures:** the +BRL 2,120 uplift (+0.9%) reported above is computed on the test set using a synthetic contact cost (BRL 1.00) and a conservative conversion value proxy (mean historical ticket). These figures are directional, not operational forecasts.
> The relevant signal is the **ratio**: the model delivers a measurable profit gain by excluding 10.3% of sends — a relationship that holds regardless of the actual cost structure applied. At iFood's real cost per contact and offer discount values, the absolute BRL impact scales accordingly.



## 14. SHAP Analysis




**How to read this Plot:**
* **Feature ranking (Y-axis):** Features are ordered from most to least impactful on model output.
* **Direction (X-axis):** SHAP values > 0 push the prediction toward conversion (target = 1); values < 0 push toward non-conversion.
* **Feature value (color):** Red dots = high feature value; blue dots = low feature value.

In [0]:
e_test.plot_shap_summary(
    estimator=trainer.estimator,
    x=x_test,
    max_display=20,
)

## 15. Final Business Recommendation

### Central Finding

Conversion is driven by **who the customer is**, not by the offer itself. Customer tenure and historical ticket size predict conversion more reliably than discount value or offer type — shifting the strategic question from *"which offer to send?"* to *"which customer to prioritise?"*

---

### Identified Opportunities

* **Score-based ranking doubles budget efficiency:** The top 30% of ranked customers (deciles 1–3) concentrate 47% of all conversions — half the opportunity at a third of the budget.

* **Long-tenured, high-ticket customers are the most responsive segment:** Not necessarily the biggest spenders today, but the most consistently engaged. Loyalty programmes that increase customer tenure may amplify propensity over time.

* **Offer duration is an underleveraged conversion driver:** Shorter offers are associated with higher conversion — likely an urgency effect. An A/B test comparing durations within the same segment would confirm this before any structural change.

* **High discounts paired with high minimum spend reduce conversion:** Moderate discounts with lower activation barriers may deliver better conversion efficiency — requires experimental validation.

* **Recency as a timing signal:** Customers who transacted or viewed an offer within the last 7 days show significantly higher propensity. A recency-triggered distribution strategy could capture this behaviour more systematically.

---

### Estimated Financial Impact (Test Set Simulation)

| Strategy | Opportunities | Conversions | Precision | ROI |
|----------|--------------|-------------|-----------|-----|
| Send-to-all (no model) | 33,988 | ~9,494 | 55.9% | ~1× |
| Model — top 30% (deciles 1–3) | ~10,196 | ~4,521 | 88%+ | — |
| Model — BRL 10k budget | 10,000 | 7,852 | 78.5% | 13.26× |

> **Assumption:** conversion value = mean historical ticket (BRL 16.62); offer discount cost not deducted. ROI figures are directional — validate with Finance before budget decisions.

---

### Risks and Limitations

* **No holdout control group:** true incrementality cannot be measured. An A/B test is required to isolate the causal effect of the offer from organic conversion.
* **Synthetic financial assumptions:** `OFFER_COST` and `avg_conversion_value` are proxies — validate with actual cost and revenue data before operational use.
* **Model reflects historical patterns:** material changes in offer mix, customer base, or platform behaviour require recalibration.

---

### Suggested Next Steps

1. **A/B holdout test** — model-targeted vs. random send — to measure true incremental lift and isolate offer-driven conversions from organic behaviour
2. **Financial validation** with Finance — replace synthetic `OFFER_COST` and `avg_conversion_value` with actual cost per offer and revenue per conversion event
3. **Duration experiment** — short vs. long offers within the same customer segment to validate the urgency effect before any structural change to the offer catalogue
4. **Feature drift monitoring** — track distribution shifts in top SHAP features (`customer_tenure_days`, `avg_ticket_before`, `days_since_last_tx`) over time; significant drift signals the need for recalibration
5. **Model recalibration** — retrain on accumulated data as offer mix, customer base, or platform behaviour evolves; establish a quarterly review cadence tied to campaign cycles
6. **Segmented models** — bogo, discount, and informational offers have distinct conversion mechanics; independent propensity models per offer type are the natural next iteration

